# Flight Delay Time Statistics Dashboard

### Material académico reproducible — Laboratorio 4.8

| | |
|---|---|
| **Tema** | Tableros interactivos: múltiples gráficos y *callbacks* con varias salidas |
| **Laboratorio original** | 4.8 *Flight Delay Time Statistics Dashboard* |



---
## 1. Objetivos de aprendizaje

Al terminar este cuaderno el estudiante será capaz de:

1. **Componer un *layout* declarativo** en Dash combinando componentes HTML (`html.H1`, `html.Div`, `html.Br`)
   y componentes *core* (`dcc.Input`, `dcc.Graph`).
2. **Controlar la disposición espacial** de los gráficos con CSS en línea y *flexbox* a través de `style={'display': 'flex'}`.
3. **Encapsular la lógica de negocio** en una función auxiliar pura (`compute_info`) separada del *layout*.
4. **Escribir un *callback* con cinco salidas** (`Output`) a partir de una sola entrada (`Input`) y comprender el
   contrato de orden entre la lista de `Output` y la lista devuelta.
5. **Servir una aplicación Dash desde un cuaderno** Jupyter sin bloquear el *kernel*.
6. **Auditar la calidad de los datos** antes de interpretar un tablero, y distinguir una muestra de una población.

**Prerrequisitos:** Python básico, `pandas` (agrupaciones y agregaciones) y nociones de HTML.

**Tiempo estimado:** 60–90 minutos (el laboratorio original estima 30 minutos; la diferencia corresponde a la
explicación conceptual y al análisis de resultados).

---
## 2. Marco conceptual

### 2.1 ¿Qué es Dash y por qué se usa para tableros?

Dash es un *framework* de Python para construir aplicaciones web analíticas sin escribir JavaScript.
Se apoya en dos ideas:

1. **El *layout* es una estructura de datos.** La interfaz no se describe con HTML escrito a mano, sino con
   árboles de objetos Python (`html.Div(...)`, `dcc.Graph(...)`). Eso permite generar la interfaz de forma
   programática y mantenerla versionada junto con el análisis.

2. **La interactividad es reactiva.** En lugar de escribir *handlers* de eventos imperativos, se declara qué
   salidas dependen de qué entradas y se delega el resto al *framework*:

$$\text{Output} = f(\text{Input})$$

   El decorador `@app.callback` registra esa relación; el navegador detecta el cambio, envía el valor al servidor,
   el servidor ejecuta la función y devuelve el nuevo estado de los componentes.

### 2.2 Arquitectura de la aplicación

```mermaid
flowchart LR
    A["Navegador<br/>(cliente Dash JS)"] -->|"1. GET /"| B["Servidor Flask<br/>app.layout -> HTML"]
    B -->|"2. DOM inicial"| A
    A -->|"3. POST /_dash-update-component<br/>input-year = 2015"| C["get_graph(2015)<br/>@app.callback"]
    C --> D["compute_info(airline_data, 2015)<br/>groupby + mean"]
    D --> E["5 figuras Plotly"]
    E -->|"4. JSON de figuras"| A
```

Los cinco `dcc.Graph` mantienen su identidad (`id`) y no se recrean: solo se reemplaza la propiedad `figure` de
cada uno. Ese es el motivo por el que un *callback* de Dash es mucho más eficiente que volver a dibujar todo el tablero.

### 2.3 Vocabulario mínimo

| Término | Significado en este laboratorio |
|---|---|
| **Componente** | Objeto Python que representa un elemento de la interfaz (`html.H1`, `dcc.Graph`, ...) |
| **`id`** | Identificador único que permite a los *callbacks* referirse a un componente |
| **`figure`** | Propiedad de `dcc.Graph` que contiene la figura de Plotly serializada |
| **Callback** | Función decorada que recalcula salidas cuando cambian sus entradas |
| **Segmento** | Agrupación visual de gráficos; en este diseño hay 3 (2 + 2 + 1 gráficos) |

---
## 3. Metodología: del Cloud IDE al entorno local

El laboratorio original asume un contenedor remoto. La adaptación es un cambio de **infraestructura**, no de lógica:

| Elemento del laboratorio | Adaptación local | Justificación |
|---|---|---|
| `python3.8 -m pip install pandas dash` | Entorno virtual `.venv` con Python 3.12 | Reproducibilidad y aislamiento de dependencias |
| `pip3 install httpx==0.20` | **Omitido** | Era un *workaround* del proxy del Cloud IDE; en local es innecesario y ese *pin* antiguo genera conflictos |
| `pd.read_csv('https://...s3.../airline_data.csv')` | Archivo local `airline_data.csv` (con descarga de respaldo) | Persistencia y ejecución sin conexión |
| `python3.8 flight_delay.py` | Servidor en un hilo dentro del cuaderno | Un cuaderno no tiene terminal interactiva para mantener el proceso vivo |
| Botón *Launch Application* + puerto | `http://127.0.0.1:8050` | El puerto por defecto de Dash es 8050; ya no hay *proxy* que mapear |

### 3.1 Nota metodológica sobre el conjunto de datos

El archivo `airline_data.csv` distribuido con el laboratorio **es una muestra** del dataset
*Airline Reporting Carrier On-Time Performance* (Bureau of Transportation Statistics).
No es la población completa: la muestra tiene 27.000 registros, mientras que el dataset original contiene
del orden de cientos de millones de vuelos.

**Consecuencia para la interpretación:** las conclusiones que se extraigan describen *esta muestra*, no la
operación aérea real. Las cifras de retraso son verosímiles, pero no deben citarse como estadísticas oficiales.
Por la misma razón, en la sección de resultados se auditará la cobertura de meses por año antes de leer los gráficos.

---
## 4. Preparación del entorno

**Instrucciones.** Antes de ejecutar el cuaderno, en la carpeta del proyecto:

```powershell
python -m venv .venv
.venv\Scripts\python.exe -m pip install pandas dash plotly ipykernel
```

Luego seleccione un *kernel* que tenga instalados `pandas`, `dash`, `plotly` y `werkzeug`.

La primera celda de código registra las versiones exactas. **Esto es parte del método**: un resultado sin la versión
del software que lo produjo no es reproducible. Ese registro permite explicar, más adelante, dos fallos reales
que aparecen al portar el laboratorio a versiones recientes.

In [42]:
# ===== 4.1 Dependencias y trazabilidad del entorno =====
import sys

import pandas as pd
import plotly
import plotly.express as px
from plotly.graph_objects import Figure

import dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State

print('Python :', sys.version.split()[0])
print('pandas :', pd.__version__)
print('plotly :', plotly.__version__)
print('dash   :', dash.__version__)


Python : 3.13.14
pandas : 2.3.2
plotly : 7.1.0
dash   : 4.4.1


---
## 5. Tarea 1 — Leer los datos

**Explicación.** Tres decisiones de lectura merecen justificación:

1. **`encoding='ISO-8859-1'`.** El archivo contiene nombres de aeropuerto con tildes y caracteres no ASCII.
   Con la codificación por defecto (`utf-8`) pandas fallaría; `ISO-8859-1` (Latin-1) nunca falla al decodificar
   bytes, y es la que documenta el laboratorio.
2. **`dtype={'Div1Airport': str, ...}`.** Los códigos de aeropuerto alterno y de cola de aeronave son identificadores,
   no números. Forzar `str` evita que pandas infiera tipos numéricos inconsistentes entre particiones del archivo
   y que se pierdan ceros a la izquierda.
3. **Ruta local con respaldo remoto.** Se replica la ruta del laboratorio, pero se usa el archivo local ya descargado
   (9,8 MB) para que el cuaderno funcione sin conexión y arranque de inmediato.

**Instrucción.** Ejecute la celda y confirme que se cargan 27.000 registros.

In [43]:
# ===== Tarea 1: lectura del conjunto de datos =====
from pathlib import Path
import urllib.request

RUTA_LOCAL = Path('airline_data.csv')
URL_ORIGEN = ('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/'
              'IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/'
              'Data%20Files/airline_data.csv')

if not RUTA_LOCAL.exists():          # respaldo: fuente original del laboratorio
    print('airline_data.csv no encontrado; descargando del origen...')
    urllib.request.urlretrieve(URL_ORIGEN, RUTA_LOCAL)

airline_data = pd.read_csv(
    RUTA_LOCAL,
    encoding='ISO-8859-1',
    dtype={'Div1Airport': str, 'Div1TailNum': str,
           'Div2Airport': str, 'Div2TailNum': str},
)

print(f'Dimensiones : {airline_data.shape[0]:,} filas x {airline_data.shape[1]} columnas')
print(f'Tamano disco: {RUTA_LOCAL.stat().st_size / 1e6:.1f} MB')
airline_data.head(3)

Dimensiones : 27,000 filas x 110 columnas
Tamano disco: 9.8 MB


,Unnamed: 0,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
0,1295781,1998,2,4,2,4,1998-04-02,AS,19930,AS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1125375,2013,2,5,13,1,2013-05-13,EV,20366,EV,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,118824,1993,3,9,25,6,1993-09-25,UA,19977,UA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5.1 Auditoría previa del conjunto de datos

**Explicación.** Un tablero es tan confiable como sus datos. Antes de dibujar nada conviene responder:
¿qué años cubre el archivo?, ¿cuántas aerolíneas informan?, ¿todas las variables de retraso están pobladas?

Las variables de retraso (`CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`)
están **desagregadas por causa** y se expresan en minutos. Son las únicas columnas del tablero que pueden
contener vacíos, y su tratamiento determina el significado de los promedios.

**Hipótesis a verificar.** La documentación del dataset original sostiene que estas columnas quedan en blanco
cuando el vuelo no sufrió un retraso atribuible a esa causa; es decir, el vacío significaría *cero*, no
*dato perdido*. Aceptarlo sin comprobarlo sería un acto de fe: la celda 5.2 lo contrasta con `ArrDelay`,
columna que sí se informa para todos los vuelos, y la 5.3 interpreta el resultado.


In [44]:
# ===== 5.1 Perfil y calidad del conjunto de datos =====
VARS_DELAY = ['CarrierDelay', 'WeatherDelay', 'NASDelay',
              'SecurityDelay', 'LateAircraftDelay']

anios = sorted(airline_data['Year'].unique())
print(f'Rango de anios   : {anios[0]} - {anios[-1]}  ({len(anios)} anios distintos)')
print(f'Aerolineas       : {airline_data["Reporting_Airline"].nunique()} codigos distintos')
print(f'Meses presentes  : {sorted(int(m) for m in airline_data["Month"].unique())}')

calidad = pd.DataFrame({
    'tipo'     : airline_data[VARS_DELAY].dtypes.astype(str),
    'no_nulos' : airline_data[VARS_DELAY].notna().sum(),
    'pct_nulos': (airline_data[VARS_DELAY].isna().mean() * 100).round(1),
})
print('\nCobertura de las variables de retraso (minutos):')
print(calidad.to_string())


Rango de anios   : 1987 - 2020  (34 anios distintos)
Aerolineas       : 33 codigos distintos
Meses presentes  : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Cobertura de las variables de retraso (minutos):
                      tipo  no_nulos  pct_nulos
CarrierDelay       float64      3057       88.7
WeatherDelay       float64      3057       88.7
NASDelay           float64      3057       88.7
SecurityDelay      float64      3057       88.7
LateAircraftDelay  float64      3057       88.7


In [45]:
# ===== 5.2 Que significa el valor ausente en las causas de retraso? =====
causas_vacias = airline_data[VARS_DELAY].isna().all(axis=1)
con_causa     = ~causas_vacias
vacios_parciales = airline_data[VARS_DELAY].isna().any(axis=1) & con_causa

print(f'Filas con las cinco causas vacias : {causas_vacias.sum():,}')
print(f'Filas con al menos una causa      : {con_causa.sum():,}')
print(f'Filas con vacios parciales        : {vacios_parciales.sum():,}')

contraste = (airline_data
             .assign(causas_vacias=causas_vacias)
             .groupby('causas_vacias')['ArrDelay']
             .agg(vuelos='count', media='mean', mediana='median', maximo='max')
             .round(2))
contraste.index = ['con causa reportada', 'sin causa reportada']

print('\nRetraso de llegada (ArrDelay, minutos) segun el estado de las causas:')
print(contraste.to_string())


Filas con las cinco causas vacias : 23,943
Filas con al menos una causa      : 3,057
Filas con vacios parciales        : 0

Retraso de llegada (ArrDelay, minutos) segun el estado de las causas:
                     vuelos  media  mediana  maximo
con causa reportada    3057  56.42     37.0   984.0
sin causa reportada   23441  -0.38     -3.0   682.0


### 5.3 Interpretación de la auditoría

La auditoría arroja cuatro hechos que condicionan toda la lectura posterior del tablero:

1. **El 88,7 % de los registros tiene las cinco causas de retraso vacías.** Solo 3.057 de los 27.000 vuelos de la
   muestra traen información de causas.
2. **No hay vacíos parciales.** Las cinco columnas están pobladas o vacías *a la vez*, en todos los casos. La
   ausencia es un estado de «todo o nada» por vuelo, coherente con que las cinco causas provengan de un mismo
   bloque del registro original.
3. **El vacío significa «no hubo retraso atribuible a esa causa», no «dato perdido».** La prueba está en el
   contraste con `ArrDelay`, que sí se informa para todos los vuelos: los registros sin causas tienen retraso de
   llegada medio de −0,38 minutos (llegan *antes* de la hora) mientras que los que sí tienen causa promedian
   56,42 minutos. Si el vacío fuera un dato perdido, ambos grupos tendrían distribuciones similares.
4. **El promedio de `compute_info` es un promedio condicional.** `mean()` ignora los `NaN`, de modo que cada punto
   de los cinco gráficos es la **duración media del retraso cuando la causa ocurrió**, calculada sobre el 11,3 % de
   los vuelos. No es el retraso medio por vuelo, y presentarlo como tal sería un error de interpretación.

> **Detalle para el análisis crítico.** El grupo sin causa reportada contiene un vuelo con 682 minutos de retraso de
> llegada y ninguna causa atribuida. Es una anomalía legítima del dato (compatible con cancelaciones, desvíos o
> errores de captura) y un buen recordatorio de que los promedios ocultan casos extremos.


---
## 6. Tarea 2 — Esqueleto del *layout*

**Explicación.** El tablero tiene tres bloques y su traducción a componentes es directa:

| Bloque del diseño | Componente Dash |
|---|---|
| Título de la aplicación | `html.H1('...')` |
| Selector de año | `html.Div(['Input Year: ', dcc.Input(...)])` |
| 5 gráficos en 3 segmentos | tres `html.Div` contenedores, cada uno con sus `dcc.Graph` |

La clave del diseño está en el estilo de los contenedores: `style={'display': 'flex'}` convierte cada segmento en un
*contenedor flexible* y coloca sus dos gráficos **lado a lado** en lugar de apilados. El tercer segmento no lleva
*flex*, sino un ancho relativo del 65 %.

*Esqueleto tal como aparece en el laboratorio (los puntos suspensivos son los valores que se completan en la Tarea 3):*

```python
app = dash.Dash(__name__)

app.layout = html.Div(children=[
    html.H1(...),                                        # titulo
    html.Div(['Input Year: ', dcc.Input(...)], style={...}),
    html.Br(),
    html.Br(),
    html.Div([                                           # segmento 1
        html.Div(...),
        html.Div(...),
    ], style={'display': 'flex'}),
    html.Div([                                           # segmento 2
        html.Div(...),
        html.Div(...),
    ], style={'display': 'flex'}),
    html.Div(..., style={'width': '65%'}),               # segmento 3
])
```

---
## 7. Tarea 3 — Completar los componentes del *layout*

**Explicación.** La especificación del laboratorio, traducida a propiedad por propiedad:

#### Título
Texto `'Flight Delay Time Statistics'`, alineado al centro, color `#503D36` y tamano de fuente 30.

#### Entrada
`id='input-year'`, `value='2010'` (valor inicial) y `type='number'` (teclado numérico y validación en el navegador).
Su `style` fija la altura en 35 px y la fuente en 30.

#### Gráficos

| Segmento | Contenedor | Componente | `id` | Propiedad que se actualizará |
|---|---|---|---|---|
| 1 | `Div(display=flex)` | `Div` #1 | `carrier-plot` | `figure` |
| 1 | | `Div` #2 | `weather-plot` | `figure` |
| 2 | `Div(display=flex)` | `Div` #1 | `nas-plot` | `figure` |
| 2 | | `Div` #2 | `security-plot` | `figure` |
| 3 | `Div(width=65%)` | — | `late-plot` | `figure` |

Observe que **todo `dcc.Graph` nace sin figura**: su contenido solo existe cuando el *callback* se ejecuta por
primera vez. Por eso los cinco `id` son la pieza que conecta *layout* y lógica.

In [46]:
# ===== Tarea 2 y 3: aplicacion y layout completo =====
app = dash.Dash(__name__)

app.layout = html.Div(children=[

    # --- Titulo ---
    html.H1('Flight Delay Time Statistics',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 30}),

    # --- Entrada: anio a analizar ---
    html.Div(['Input Year: ',
              dcc.Input(id='input-year', value='2010', type='number',
                        style={'height': '35px', 'font-size': 30})],
             style={'font-size': 30}),
    html.Br(),
    html.Br(),

    # --- Segmento 1: retraso por aerolinea y por clima ---
    html.Div([
        html.Div(dcc.Graph(id='carrier-plot')),
        html.Div(dcc.Graph(id='weather-plot')),
    ], style={'display': 'flex'}),

    # --- Segmento 2: retraso del sistema aereo nacional y por seguridad ---
    html.Div([
        html.Div(dcc.Graph(id='nas-plot')),
        html.Div(dcc.Graph(id='security-plot')),
    ], style={'display': 'flex'}),

    # --- Segmento 3: retraso por aeronave tardia ---
    html.Div(dcc.Graph(id='late-plot'), style={'width': '65%'}),
])


def recorrer(componente):
    """Devuelve el id de todos los componentes con id dentro del arbol del layout."""
    encontrados = []
    identificador = getattr(componente, 'id', None)
    if identificador:
        encontrados.append(identificador)
    hijos = getattr(componente, 'children', None)
    if isinstance(hijos, (list, tuple)):
        for hijo in hijos:
            encontrados += recorrer(hijo)
    elif hijos is not None and not isinstance(hijos, str):
        encontrados += recorrer(hijos)
    return encontrados


print('Aplicacion creada :', type(app).__name__)
print('Componentes con id:', sorted(recorrer(app.layout)))


Aplicacion creada : Dash
Componentes con id: ['carrier-plot', 'input-year', 'late-plot', 'nas-plot', 'security-plot', 'weather-plot']


---
## 8. Tarea 4 — Función auxiliar `compute_info`

**Explicación.** Esta función concentra *todo* el cálculo y no conoce nada de Dash: recibe un `DataFrame` y un año,
y devuelve cinco tablas. Esa separación —**cálculo puro** frente a **capa de presentación**— es la razón por la que
el *callback* final cabe en pocas líneas y por la que `compute_info` puede probarse sin levantar el servidor.

La cadena de operaciones de cada tabla es:

1. `groupby(['Month', 'Reporting_Airline'])` — agrupa por mes y aerolínea (la unidad de análisis del tablero).
2. `['CarrierDelay']` — selecciona una única variable de retraso.
3. `.mean()` — promedio de las observaciones del grupo, **ignorando los `NaN`** (por defecto `skipna=True`).
4. `.reset_index()` — devuelve `Month` y `Reporting_Airline` como columnas, formato que `px.line` espera
   para `x='Month'`, `color='Reporting_Airline'`.

> **Advertencia metodológica.** El promedio ignora los `NaN`. Si en la muestra la mayoría de vuelos de una aerolínea
> no registró retraso por clima, el `mean()` de `WeatherDelay` se calcula solo sobre los vuelos que *sí* tuvieron
> esa causa. El valor es entonces una **duración media del retraso cuando ocurre**, no un retraso medio por vuelo.
> Leer el gráfico como lo segundo es un error de interpretación frecuente.

In [47]:
# ===== Tarea 4: funcion auxiliar de calculo =====
def compute_info(airline_data, entered_year):
    """Promedios mensuales de retraso por aerolinea para un anio dado.

    Argumentos:
        airline_data: DataFrame con el historico de vuelos.
        entered_year: anio seleccionado por el usuario.

    Devuelve:
        Cinco DataFrames (carrier, weather, NAS, security, late aircraft),
        cada uno con columnas Month, Reporting_Airline y la variable de retraso.
    """
    df = airline_data[airline_data['Year'] == int(entered_year)]

    avg_car     = df.groupby(['Month', 'Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Month', 'Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS     = df.groupby(['Month', 'Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec     = df.groupby(['Month', 'Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late    = df.groupby(['Month', 'Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()

    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late


prueba = compute_info(airline_data, 2010)
print('Tablas devueltas :', len(prueba))
print('Forma de cada una:', [t.shape for t in prueba])
prueba[0].head(3)

Tablas devueltas : 5
Forma de cada una: [(195, 3), (195, 3), (195, 3), (195, 3), (195, 3)]


,Month,Reporting_Airline,CarrierDelay
0,1,9E,7.0
1,1,AA,13.0
2,1,B6,NaN


---
## 9. Tareas 5 y 6 — El *callback* de cinco salidas

**Explicación.** El decorador establece el contrato entre interfaz y cálculo:

```python
@app.callback([
    Output(component_id='carrier-plot',  component_property='figure'),
    Output(component_id='weather-plot',  component_property='figure'),
    Output(component_id='nas-plot',      component_property='figure'),
    Output(component_id='security-plot', component_property='figure'),
    Output(component_id='late-plot',     component_property='figure'),
], Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    ...
```

Tres reglas que conviene memorizar:

1. **El orden importa.** La lista devuelta por la función se asigna posicionalmente a la lista de `Output`.
   Si se permutan, los gráficos aparecen intercambiados sin ningún mensaje de error.
2. **Cada `Output` es un par (componente, propiedad).** `component_id` es el `id` del `layout`;
   `component_property` es la propiedad que el *callback* sobrescribe (`figure` en los cinco casos).
3. **El `Input` entrega el valor actual** de `input-year`. Con `type='number'`, Dash envía un número o `None`
   (por ejemplo, si el usuario borra el campo).

**Defensa ante entradas inválidas.** El punto 3 obliga a blindar la función: `int(None)` lanza `TypeError` y
`int('')` lanza `ValueError`. Sin una guarda, vaciar la casilla rompe el *callback* y el tablero queda congelado.
La guarda devuelve cinco figuras vacías con un mensaje.

> **Detalle real de portabilidad.** La forma natural de crear esa figura vacía, `px.line(title='...')` sin datos,
> **falla en Plotly 7** con `TypeError: object of type 'NoneType' has no len()`. Por eso se usa `go.Figure()`.
> Este comportamiento no aparece documentado en el laboratorio, que fue escrito para Plotly 5.

In [48]:
# ===== Tareas 5 y 6: callback con cinco salidas =====
@app.callback([
    Output(component_id='carrier-plot',  component_property='figure'),
    Output(component_id='weather-plot',  component_property='figure'),
    Output(component_id='nas-plot',      component_property='figure'),
    Output(component_id='security-plot', component_property='figure'),
    Output(component_id='late-plot',     component_property='figure'),
], Input(component_id='input-year', component_property='value'))
def get_graph(entered_year):
    """Devuelve las cinco figuras del tablero para el anio seleccionado."""

    # Guarda: la casilla puede quedar vacia (None) o con texto no numerico.
    try:
        int(entered_year)
    except (TypeError, ValueError):
        vacia = Figure()
        vacia.update_layout(title='Introduzca un anio valido (2010-2020)',
                            xaxis={'visible': False}, yaxis={'visible': False})
        return [vacia] * 5

    # Calculo delegado en la funcion auxiliar
    avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, entered_year)

    carrier_fig  = px.line(avg_car, x='Month', y='CarrierDelay', color='Reporting_Airline',
                           title='Average carrier delay time (minutes) by airline')
    weather_fig  = px.line(avg_weather, x='Month', y='WeatherDelay', color='Reporting_Airline',
                           title='Average weather delay time (minutes) by airline')
    nas_fig      = px.line(avg_NAS, x='Month', y='NASDelay', color='Reporting_Airline',
                           title='Average NAS delay time (minutes) by airline')
    sec_fig      = px.line(avg_sec, x='Month', y='SecurityDelay', color='Reporting_Airline',
                           title='Average security delay time (minutes) by airline')
    late_fig     = px.line(avg_late, x='Month', y='LateAircraftDelay', color='Reporting_Airline',
                           title='Average late aircraft delay time (minutes) by airline')

    return [carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]


print('Salidas registradas en el callback:')
for clave in sorted(app.callback_map):
    print('  -', clave)

Salidas registradas en el callback:
  - ..carrier-plot.figure...weather-plot.figure...nas-plot.figure...security-plot.figure...late-plot.figure..


### 9.1 Verificar el *callback* sin abrir el navegador

**Explicación.** Un *callback* de Dash es, por dentro, una función Python normal: el decorador la registra y la
devuelve sin envolverla. Por tanto se puede invocar directamente y **probar la lógica antes de servir la aplicación**.
Esta es la técnica de prueba más barata disponible y conviene aplicarla siempre, incluidos los casos límite:

| Escenario | Entrada | Resultado esperado |
|---|---|---|
| Año con datos | `'2010'` | 5 figuras con series por aerolínea |
| Año reciente | `'2020'` | 5 figuras con menos meses cubiertos |
| Año fuera del rango pedido | `'1990'` | 5 figuras (el dataset sí tiene 1990; el enunciado pide 2010-2020) |
| Casilla vacía | `''` o `None` | 5 figuras vacías con aviso, **sin excepción** |

In [49]:
# ===== 9.1 Prueba del callback como funcion pura =====
for escenario in ['2010', '2020', '1990', '']:
    try:
        figuras = get_graph(escenario)
        detalle = [f'{len(f.data)} series' for f in figuras]
        print(f'get_graph({escenario!r:>6}) -> OK   :', detalle)
    except Exception as error:                      # noqa: BLE001
        print(f'get_graph({escenario!r:>6}) -> FALLO:', type(error).__name__, error)

get_graph('2010') -> OK   : ['18 series', '18 series', '18 series', '18 series', '18 series']
get_graph('2020') -> OK   : ['17 series', '17 series', '17 series', '17 series', '17 series']
get_graph('1990') -> OK   : ['12 series', '12 series', '12 series', '12 series', '12 series']
get_graph(    '') -> OK   : ['0 series', '0 series', '0 series', '0 series', '0 series']


In [50]:
# ===== 9.2 Vista previa de las figuras generadas para 2010 =====
from IPython.display import display

figuras = get_graph('2010')
etiquetas = ['Carrier', 'Weather', 'NAS', 'Security', 'Late aircraft']

print('Series por figura:', {e: len(f.data) for e, f in zip(etiquetas, figuras)})

for etiqueta, figura in zip(etiquetas[:2], figuras[:2]):   # las otras tres son analogas
    figura.update_layout(height=320, margin={'l': 40, 'r': 10, 't': 50, 'b': 30},
                         title=f'{etiqueta} delay - 2010')
    display(figura)

Series por figura: {'Carrier': 18, 'Weather': 18, 'NAS': 18, 'Security': 18, 'Late aircraft': 18}


---
## 10. Puesta en marcha de la aplicación

**Explicación.** Fuera del cuaderno la aplicación se lanza con `app.run()`. Dentro de un cuaderno esa llamada es
inadecuada: **bloquea el *kernel*** y el resto de celdas nunca se ejecutaría.

La solución es montar el servidor en un **hilo demonio**:

1. `werkzeug.serving.make_server(...)` construye el servidor WSGI **sin arrancarlo** (a diferencia de `app.run()`).
2. El servidor se ejecuta en un `Thread` independiente con `daemon=True`, de modo que muere con el *kernel*.
3. `threaded=True` permite atender en paralelo las peticiones de la página, los *assets* y los *callbacks*.
4. El *kernel* queda libre y sigue ejecutando celdas.

La celda espera de forma activa (*polling*) hasta que el puerto responde, para no mostrar un `iframe` vacío,
y reutiliza el servidor si ya estaba levantado, de manera que **volver a ejecutar la celda no provoca un error de
puerto ocupado**.

**Instrucción.** Ejecute la celda y abra la dirección indicada. Si el `iframe` no se renderiza (algunos visores de
cuadernos bloquean contenido de `localhost` por políticas de seguridad), use el enlace de la salida.

In [51]:
# ===== 10.1 Servidor Dash en un hilo, dentro del cuaderno =====
import socket
import threading
import time

from IPython.display import IFrame, Markdown, display
from werkzeug.serving import make_server

PUERTO = 8050
# El registro se conserva si la celda se reejecuta: asi los tableros ya abiertos
# en otros puertos siguen siendo accesibles y detenibles.
_servidores = _servidores if '_servidores' in globals() else {}


def puerto_activo(puerto):
    """True si algo responde en 127.0.0.1:<puerto>."""
    with socket.socket() as conexion:
        return conexion.connect_ex(('127.0.0.1', puerto)) == 0


def lanzar_dashboard(app, puerto=PUERTO, alto=820):
    """Sirve una aplicacion Dash en un hilo demonio y devuelve el marco embebido.

    Admite varias llamadas con puertos distintos y detecta el caso delicado: si
    se reejecuta la celda del layout, la aplicacion es un objeto NUEVO y el
    servidor anterior estaria sirviendo una version obsoleta. En ese caso se
    apaga el servidor viejo y se levanta uno nuevo.
    """
    registrado = _servidores.get(puerto)

    if registrado is not None and registrado['app'] is not app:
        registrado['servidor'].shutdown()
        del _servidores[puerto]
        print(f'El puerto {puerto} servia una version anterior del tablero: se reinicia.')

    if puerto in _servidores:
        print(f'El puerto {puerto} ya sirve esta misma aplicacion; se reutiliza.')
    elif puerto_activo(puerto):
        print(f'El puerto {puerto} ya responde y no lo gestiona el cuaderno '
              f'(lo inicio una version anterior de esta celda).')
    else:
        servidor = make_server('127.0.0.1', puerto, app.server, threaded=True)
        threading.Thread(target=servidor.serve_forever, daemon=True).start()
        _servidores[puerto] = {'servidor': servidor, 'app': app}
        for _ in range(50):                 # hasta 5 s de espera activa
            if puerto_activo(puerto):
                break
            time.sleep(0.1)

    print(f'Dashboard disponible en http://127.0.0.1:{puerto}/')
    return IFrame(f'http://127.0.0.1:{puerto}/', width='100%', height=alto)


def detener_dashboard(puerto=None):
    """Detiene un tablero concreto o todos los que gestiona el cuaderno."""
    objetivos = [puerto] if puerto is not None else list(_servidores)
    if not objetivos:
        print('No hay servidores gestionados por el cuaderno.')
        return
    for p in objetivos:
        registrado = _servidores.pop(p, None)
        if registrado is not None:
            registrado['servidor'].shutdown()
            print(f'Servidor del puerto {p} detenido; el puerto quedo libre.')


display(lanzar_dashboard(app))
display(Markdown(f'**[Abrir el dashboard en una pestana del navegador](http://127.0.0.1:{PUERTO}/)**'))


El puerto 8050 ya responde y no lo gestiona el cuaderno (lo inicio una version anterior de esta celda).
Dashboard disponible en http://127.0.0.1:8050/


**[Abrir el dashboard en una pestana del navegador](http://127.0.0.1:8050/)**

**Instrucción de uso.** Con el tablero abierto, cambie el año en el selector y observe cómo las cinco gráficas se
recalculan sin recargar la página. Cada cambio produce una petición `POST /_dash-update-component` en el servidor:
esa es la evidencia de que el *callback* se está ejecutando.

Al terminar, ejecute `detener_dashboard()` para liberar el puerto 8050.

---
## 11. Resultados

**Explicación.** Antes de juzgar el contenido de los gráficos hay que saber cuánta información respalda cada año.
La siguiente tabla resume, para el rango 2010-2020 que pide el enunciado, cuántas aerolíneas informan, cuántos
meses están cubiertos y cuál es el retraso medio de las dos causas de mayor peso.

In [52]:
# ===== 11.1 Perfil del rango 2010-2020 =====
rango = airline_data[airline_data['Year'].between(2010, 2020)]

perfil_anual = (rango
    .groupby('Year')
    .agg(aerolineas      = ('Reporting_Airline', 'nunique'),
         registros       = ('Reporting_Airline', 'size'),
         meses_cubiertos = ('Month', 'nunique'),
         retraso_carrier = ('CarrierDelay', 'mean'),
         retraso_late    = ('LateAircraftDelay', 'mean'))
    .round(2))

perfil_anual

,aerolineas,registros,meses_cubiertos,retraso_carrier,retraso_late
Year,,,,,
2010,18,950,12,18.33,21.26
2011,16,919,12,11.47,22.96
2012,15,791,12,22.79,26.01
2013,16,869,12,15.29,18.44
2014,14,853,12,14.04,31.32
2015,14,794,12,22.87,20.66
2016,12,758,12,17.11,22.17
2017,12,761,12,22.34,28.01
2018,18,961,12,13.53,28.53


In [53]:
# ===== 11.2 Cuantos registros aporta cada aerolinea en 2010 y en 2020 =====
for anio in (2010, 2020):
    conteo = (airline_data[airline_data['Year'] == anio]
              .groupby('Reporting_Airline')
              .size()
              .sort_values(ascending=False))
    print(f'--- {anio}: {len(conteo)} aerolineas, '
          f'de {conteo.min()} a {conteo.max()} registros por aerolinea ---')
    print(conteo.head(5).to_string(), '\n')

--- 2010: 18 aerolineas, de 9 a 166 registros por aerolinea ---
Reporting_Airline
WN    166
DL    102
AA     99
OO     92
MQ     76 

--- 2020: 17 aerolineas, de 2 a 39 registros por aerolinea ---
Reporting_Airline
AA    39
DL    31
WN    29
OO    27
UA    20 



### 11.3 Hallazgos

Las celdas anteriores producen evidencia suficiente para sostener cinco afirmaciones. Todas ellas son
verificables reejecutando el cuaderno.

1. **La cobertura por año es muy desigual.** El número de aerolíneas informantes oscila entre 12 (2016 y 2017)
   y 18 (2010 y 2018); los registros anuales van de 961 (2018) a 232 (2020), porque **2020 solo cubre 3 meses**
   frente a los 12 de los demás años del rango.
2. **El esfuerzo de muestreo por aerolínea es aún más desigual que el anual.** En 2010 las aerolíneas aportan
   entre 9 y 166 registros (`WN` 166, `DL` 102, `AA` 99); en 2020, entre **2 y 39**. Como `compute_info` promedia
   dentro de cada celda mes × aerolínea, en 2020 hay promedios sostenidos por muy pocos vuelos y son, por tanto,
   inestables. Comparar 2010 con 2020 en el tablero es una comparación **mal controlada**, aunque el gráfico la
   muestre con la misma apariencia.
3. **La composición de aerolíneas cambia entre años.** Al pasar de 2010 a 2015 desaparecen códigos como `CO`,
   `FL`, `XE` y `YV` y aparecen `NK` y `VX`, por fusiones y entradas al mercado. En un gráfico de líneas con
   `color='Reporting_Airline'` esto se manifiesta como líneas que empiezan o terminan antes: **no es un error del
   código**, es la realidad del panel de aerolíneas.
4. **El retraso por aeronave tardía tiende a ser el mayor.** En 9 de los 11 años el promedio de
   `LateAircraftDelay` supera al de `CarrierDelay` (por ejemplo, en 2014: 31,32 frente a 14,04 minutos). Las
   excepciones son 2015 y 2016, donde el retraso imputable a la aerolínea es mayor. El retraso por seguridad es
   el más pequeño en todos los años y su eje Y resulta prácticamente plano.
5. **Cada punto del gráfico es un promedio condicional, no un promedio por vuelo.** Como se estableció en la
   sección 5.3, el promedio se calcula solo sobre los vuelos que sufrieron esa causa (11,3 % de los registros).
   Además, existen celdas mes × aerolínea sin ningún vuelo con causa reportada: allí el promedio es `NaN` y
   Plotly **corta la línea** en lugar de dibujar un cero. Los huecos en las series son eso, y no datos faltantes
   de la aplicación.


---
## 12. Dashboard 2 — Explorador con filtros combinados

**Objetivo.** El tablero del laboratorio solo permite cambiar el año. Aquí se construye otro que combina
**cinco controles** y **siete salidas**, para mostrar cómo crece la interactividad sin que crezca la complejidad
del código.

### 12.0 Qué se podrá hacer

| Control | Componente | Efecto |
|---|---|---|
| Aerolíneas | `dcc.Dropdown(multi=True)` | Filtra por una o varias aerolíneas a la vez |
| Rango de años | `dcc.RangeSlider` | Acota el periodo 2010-2020 con dos manejadores |
| Tipo de gráfico | `dcc.RadioItems` | Alterna entre líneas y barras **sobre los mismos datos** |
| Causas | `dcc.Checklist` | Elige cuáles de las cinco causas se comparan |
| Descarga | `html.Button` + `dcc.Download` | Exporta el subconjunto filtrado a CSV |

Como salidas: **cuatro indicadores** recalculados, un gráfico envuelto en `dcc.Loading` (indicador de progreso
mientras el servidor calcula), una **tabla ordenable y filtrable** (`dash_table.DataTable`) y el archivo
descargable.

### 12.1 Tres conceptos nuevos de *callback*

1. **`Input` frente a `State`.** Un `Input` **dispara** el *callback* cuando cambia; un `State` entrega el valor
   actual **sin dispararlo**. Los filtros van como `Input` (respuesta inmediata), pero el botón de descarga los
   recibe como `State`: si fueran `Input`, se generaría un archivo con cada clic del deslizador.
2. **`prevent_initial_call=True`.** Sin ese argumento, el *callback* de descarga se ejecutaría al cargar la
   página y el navegador pediría un archivo que nadie solicitó.
3. **Una propiedad, un solo escritor.** Dash prohíbe por diseño que dos *callbacks* escriban en la misma
   propiedad, porque el resultado deja de ser determinista. Si hiciera falta, se declara con
   `allow_duplicate=True` de forma explícita.

> **Nota de arquitectura.** Los cinco controles alimentan **un único** *callback* con cuatro salidas. Agrupar por
> bloque lógico (filtros → resultados) es lo que mantiene el código legible; un *callback* por gráfico obligaría a
> repetir el filtrado del `DataFrame` cinco veces.


In [54]:
# ===== 12.2 Datos de trabajo y vocabulario de los tableros interactivos =====
ANIO_MIN, ANIO_MAX = 2010, 2020

# Etiquetas legibles para el usuario final.
# OJO: ninguna debe coincidir con 'Aerolinea' ni con 'Anio', porque esas dos son
# nombres de columna de la tabla agregada. Si coinciden, pandas crea columnas
# duplicadas y to_dict('records') descarta una de ellas en silencio.
ETIQUETAS_CAUSA = {
    'CarrierDelay': 'Retraso de la aerolinea',
    'WeatherDelay': 'Clima',
    'NASDelay': 'Sistema aereo nacional',
    'SecurityDelay': 'Seguridad',
    'LateAircraftDelay': 'Aeronave tardia',
}

# Subconjunto del rango que pide el enunciado
vuelos = airline_data[airline_data['Year'].between(ANIO_MIN, ANIO_MAX)].copy()

# Subconjunto con alguna causa reportada: es el unico sobre el que tiene sentido
# promediar las causas (ver la advertencia de la seccion 5.3)
vuelos_con_causa = vuelos[vuelos[VARS_DELAY].notna().any(axis=1)].copy()
vuelos_con_causa['periodo'] = pd.to_datetime(
    dict(year=vuelos_con_causa['Year'].astype(int),
         month=vuelos_con_causa['Month'].astype(int), day=1))

AEROLINEAS = sorted(vuelos['Reporting_Airline'].unique())
OPCIONES_AEROLINEA = [{'label': a, 'value': a} for a in AEROLINEAS]
OPCIONES_CAUSA = [{'label': ETIQUETAS_CAUSA[c], 'value': c} for c in VARS_DELAY]

print(f'Vuelos {ANIO_MIN}-{ANIO_MAX}            : {len(vuelos):,}')
print(f'  con alguna causa reportada : {len(vuelos_con_causa):,} '
      f'({len(vuelos_con_causa) / len(vuelos):.1%})')
print(f'Aerolineas disponibles       : {len(AEROLINEAS)}')
print(f'Rango temporal de la serie   : {vuelos_con_causa["periodo"].min():%Y-%m} a '
      f'{vuelos_con_causa["periodo"].max():%Y-%m}')


Vuelos 2010-2020            : 8,914
  con alguna causa reportada : 1,631 (18.3%)
Aerolineas disponibles       : 22
Rango temporal de la serie   : 2010-01 a 2020-03


In [55]:
# ===== 12.3 Dashboard 2: layout con cinco controles =====
app2 = dash.Dash('explorador_retrasos')


def tarjeta(titulo, valor):
    """Tarjeta de indicador (KPI) reutilizable por los dos tableros nuevos."""
    return html.Div([
        html.Div(titulo, style={'fontSize': 13, 'color': '#666666'}),
        html.Div(valor, style={'fontSize': 24, 'fontWeight': 'bold', 'color': '#503D36'}),
    ], style={'flex': '1', 'border': '1px solid #dddddd', 'borderRadius': '8px',
              'padding': '10px 14px', 'backgroundColor': '#fafafa'})


app2.layout = html.Div([

    html.H1('Explorador de retrasos 2010-2020',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 30}),

    # --- Fila de filtros ---
    html.Div([
        html.Div([
            html.Label('Aerolineas (seleccion multiple)'),
            dcc.Dropdown(id='d2-aerolineas', options=OPCIONES_AEROLINEA,
                         value=['AA', 'DL', 'UA', 'WN'], multi=True),
        ], style={'width': '34%', 'padding': '0 1%'}),

        html.Div([
            html.Label('Rango de anios'),
            dcc.RangeSlider(id='d2-anios', min=ANIO_MIN, max=ANIO_MAX, step=1,
                            value=[ANIO_MIN, ANIO_MAX],
                            marks={a: str(a) for a in range(ANIO_MIN, ANIO_MAX + 1, 2)}),
        ], style={'width': '34%', 'padding': '22px 1% 0 1%'}),

        html.Div([
            html.Label('Tipo de grafico'),
            dcc.RadioItems(id='d2-tipo',
                           options=[{'label': ' Lineas', 'value': 'line'},
                                    {'label': ' Barras', 'value': 'bar'}],
                           value='bar', inline=True),
        ], style={'width': '28%', 'padding': '0 1%'}),
    ], style={'display': 'flex', 'alignItems': 'flex-start'}),

    html.Div([
        html.Label('Causas de retraso a comparar'),
        dcc.Checklist(id='d2-causas', options=OPCIONES_CAUSA, inline=True,
                      value=['CarrierDelay', 'NASDelay', 'LateAircraftDelay']),
    ], style={'padding': '12px 1% 4px 1%'}),

    # --- Indicadores recalculados por el callback ---
    html.Div(id='d2-kpis', style={'display': 'flex', 'gap': '12px', 'padding': '8px 1%'}),

    # --- Grafico principal, con indicador de progreso ---
    html.Div(dcc.Loading(dcc.Graph(id='d2-grafico', style={'height': '55vh'})),
             style={'padding': '0 1%'}),

    # --- Descarga del subconjunto filtrado ---
    html.Div([
        html.Button('Descargar el detalle en CSV', id='d2-boton-csv', n_clicks=0,
                    style={'padding': '8px 16px', 'cursor': 'pointer'}),
        dcc.Download(id='d2-archivo'),
    ], style={'padding': '10px 1%'}),

    # --- Tabla ordenable y filtrable ---
    html.Div([
        html.H3('Detalle por aerolinea y anio (ordene y filtre con las cabeceras)'),
        dash_table.DataTable(
            id='d2-tabla',
            sort_action='native',
            filter_action='native',
            page_size=8,
            style_table={'overflowX': 'auto'},
            style_cell={'textAlign': 'left', 'padding': '6px', 'fontSize': 13},
            style_header={'backgroundColor': '#503D36', 'color': 'white', 'fontWeight': 'bold'},
        ),
    ], style={'padding': '0 1% 24px 1%'}),
])

print('Dashboard 2 -> puerto 8051')
print('Componentes con id:', sorted(recorrer(app2.layout)))


Dashboard 2 -> puerto 8051
Componentes con id: ['d2-aerolineas', 'd2-anios', 'd2-archivo', 'd2-boton-csv', 'd2-causas', 'd2-grafico', 'd2-kpis', 'd2-tabla', 'd2-tipo']


In [56]:
# ===== 12.4 Dashboard 2: los dos callbacks =====
@app2.callback(
    Output('d2-kpis', 'children'),
    Output('d2-grafico', 'figure'),
    Output('d2-tabla', 'data'),
    Output('d2-tabla', 'columns'),
    Input('d2-aerolineas', 'value'),
    Input('d2-anios', 'value'),
    Input('d2-tipo', 'value'),
    Input('d2-causas', 'value'),
)
def actualizar_explorador(aerolineas, rango_anios, tipo, causas):
    """Recalcula indicadores, grafico y tabla a partir de los cuatro filtros."""
    aerolineas = aerolineas or []          # Dropdown sin seleccion entrega None
    causas = causas or []                  # Checklist sin marcas entrega []

    datos = vuelos[vuelos['Reporting_Airline'].isin(aerolineas)
                   & vuelos['Year'].between(rango_anios[0], rango_anios[1])]
    con_causa_local = datos[datos[VARS_DELAY].notna().any(axis=1)]

    # --- 1. Indicadores ---
    retraso_llegada = datos['ArrDelay'].mean() if len(datos) else float('nan')
    proporcion = f'{len(con_causa_local) / len(datos):.0%}' if len(datos) else 's/d'
    kpis = [
        tarjeta('Vuelos analizados', f'{len(datos):,}'),
        tarjeta('Retraso medio de llegada', f'{retraso_llegada:,.1f} min'),
        tarjeta('Vuelos con causa reportada', proporcion),
        tarjeta('Aerolineas en el filtro', f'{datos["Reporting_Airline"].nunique()}'),
    ]

    # --- 2. Grafico: duracion media por causa y anio ---
    if causas and len(datos):
        resumen = datos.groupby('Year')[causas].mean().reset_index()
        comunes = dict(
            data_frame=resumen, x='Year', y=causas,
            labels={'value': 'Minutos', 'variable': 'Causa', 'Year': 'Anio'},
            title=f'Duracion media del retraso por causa ({rango_anios[0]}-{rango_anios[1]})')
        # OJO: px.line NO acepta barmode en Plotly 7 (TypeError: line() got an
        # unexpected keyword argument). Cada constructor recibe solo lo suyo.
        if tipo == 'bar':
            figura = px.bar(barmode='group', **comunes)
        else:
            figura = px.line(markers=True, **comunes)
    else:
        figura = Figure()
        figura.update_layout(title='Seleccione al menos una causa y una aerolinea',
                             xaxis={'visible': False}, yaxis={'visible': False})

    figura.update_layout(height=420, yaxis_title='Minutos', legend_title_text='Causa')
    figura.for_each_trace(
        lambda traza: traza.update(name=ETIQUETAS_CAUSA.get(traza.name, traza.name)))

    # --- 3. Tabla con columnas dinamicas ---
    if len(datos):
        agrupado = (datos.groupby(['Reporting_Airline', 'Year'])
                    .agg(Vuelos=('ArrDelay', 'size'),
                         **{'Retraso llegada': ('ArrDelay', 'mean')},
                         **{ETIQUETAS_CAUSA[c]: (c, 'mean') for c in causas})
                    .round(2)
                    .reset_index()
                    .rename(columns={'Reporting_Airline': 'Aerolinea', 'Year': 'Anio'}))
        tabla = agrupado.to_dict('records')
        columnas = [{'name': columna, 'id': columna} for columna in agrupado.columns]
    else:
        tabla, columnas = [], []

    return kpis, figura, tabla, columnas


@app2.callback(
    Output('d2-archivo', 'data'),
    Input('d2-boton-csv', 'n_clicks'),
    State('d2-aerolineas', 'value'),
    State('d2-anios', 'value'),
    State('d2-causas', 'value'),
    prevent_initial_call=True,
)
def descargar_csv(n_clicks, aerolineas, rango_anios, causas):
    """Devuelve el CSV del subconjunto filtrado.

    Los filtros llegan por State: cambiar un deslizador NO dispara esta funcion;
    solo lo hace el boton, que es exactamente lo que se quiere.
    """
    aerolineas = aerolineas or []
    columnas = ['Year', 'Month', 'Reporting_Airline', 'ArrDelay'] + list(causas or [])
    datos = vuelos[vuelos['Reporting_Airline'].isin(aerolineas)
                   & vuelos['Year'].between(rango_anios[0], rango_anios[1])]
    return dcc.send_data_frame(datos[columnas].to_csv, 'vuelos_filtrados.csv', index=False)


print('Callbacks registrados en el dashboard 2:')
for clave in sorted(app2.callback_map):
    print('  -', clave[:70], '...')


Callbacks registrados en el dashboard 2:
  - ..d2-kpis.children...d2-grafico.figure...d2-tabla.data...d2-tabla.colu ...
  - d2-archivo.data ...


In [ ]:
# ===== 12.5 Prueba de los callbacks y puesta en marcha del dashboard 2 =====
# Se prueban LAS DOS ramas del grafico: barras y lineas. Probar solo una dejo
# pasar un error 500 real (px.line no acepta barmode en Plotly 7).
kpis, figura, tabla, columnas = actualizar_explorador(
    ['AA', 'DL'], [2015, 2020], 'bar', ['CarrierDelay', 'LateAircraftDelay'])
print('Tarjetas de indicador :', len(kpis))
print('Rama BARRAS           :', [traza.name for traza in figura.data],
      '| tipos:', sorted({traza.type for traza in figura.data}))

_, figura_lineas, _, _ = actualizar_explorador(
    ['AA', 'DL'], [2015, 2020], 'line', ['CarrierDelay', 'LateAircraftDelay'])
print('Rama LINEAS           :', [traza.name for traza in figura_lineas.data],
      '| tipos:', sorted({traza.type for traza in figura_lineas.data}))

print('Filas de la tabla     :', len(tabla))
print('Columnas de la tabla  :', [c['name'] for c in columnas])
print('Primera fila          :', tabla[0] if tabla else 'sin datos')

# --- Caso limite: el usuario quita todos los filtros ---
_, figura_vacia, tabla_vacia, columnas_vacias = actualizar_explorador([], [2015, 2016], 'line', [])
print()
print('Sin aerolineas ni causas ->', figura_vacia.layout.title.text,
      '| tabla:', tabla_vacia, '| columnas:', columnas_vacias)

display(lanzar_dashboard(app2, puerto=8051))
display(Markdown('**[Abrir el explorador en una pestana](http://127.0.0.1:8051/)**'))


Tarjetas de indicador : 4
Rama BARRAS           : ['Retraso de la aerolinea', 'Aeronave tardia'] | tipos: ['bar']
Rama LINEAS           : ['Retraso de la aerolinea', 'Aeronave tardia'] | tipos: ['scatter']
Filas de la tabla     : 12
Columnas de la tabla  : ['Aerolinea', 'Anio', 'Vuelos', 'Retraso llegada', 'Retraso de la aerolinea', 'Aeronave tardia']
Primera fila          : {'Aerolinea': 'AA', 'Anio': 2015, 'Vuelos': 95, 'Retraso llegada': 6.76, 'Retraso de la aerolinea': 30.0, 'Aeronave tardia': 25.25}

Sin aerolineas ni causas -> Seleccione al menos una causa y una aerolinea | tabla: [] | columnas: []
El puerto 8051 servia una version anterior del tablero: se reinicia.
Dashboard disponible en http://127.0.0.1:8051/


**[Abrir el explorador en una pestana](http://127.0.0.1:8051/)**

127.0.0.1 - - [22/Sep/2026 20:17:56] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_4_1m1790119530.12.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_4_1m1790119530.8.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/deps/react@18.v4_4_1m1790119530.3.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_4_1m1790119530.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_4_1m1790119530.3.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_4_1m1790119530.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:17:56] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v4_4_1m1790119530.js HTTP/

In [58]:
# --- celda temporal de mantenimiento: se elimina al terminar la verificacion ---
if '_servidor' in globals() and _servidor is not None:
    _servidor.shutdown()
    _servidor = None
    print('Servidor obsoleto del puerto 8051 detenido.')

display(lanzar_dashboard(app2, puerto=8051))


El puerto 8051 ya sirve esta misma aplicacion; se reutiliza.
Dashboard disponible en http://127.0.0.1:8051/


**Instrucciones de uso del explorador.**

1. Abra <http://127.0.0.1:8051/> y compruebe los cuatro indicadores iniciales.
2. Quite una aerolínea del desplegable: los indicadores y la tabla se reducen de inmediato.
3. Arrastre un manejador del control de años: el gráfico y la tabla siguen el nuevo rango.
4. Cambie de **barras** a **líneas**: son los mismos datos, otra lectura.
5. Desmarque todas las causas y observe el mensaje de aviso en el gráfico (el *callback* no falla).
6. Pulse **Descargar el detalle en CSV** y abra el archivo: contiene exactamente el subconjunto filtrado.

> **Qué demuestra.** Que un tablero analítico no es «un gráfico en la web», sino **una consulta parametrizada
> en tiempo real**: los cinco controles definen el filtro, y todas las salidas —indicadores, gráfico, tabla y
> archivo— se derivan de la misma consulta.


---
## 13. Dashboard 3 — Interacción cruzada: del clic al detalle

**Objetivo.** Aquí el usuario no rellena formularios: **interactúa con los gráficos**. Es el tipo de interacción
que convierte un tablero en una herramienta de exploración.

### 13.0 Qué se podrá hacer

| Acción del usuario | Propiedad que se activa | Respuesta del tablero |
|---|---|---|
| Clic en una barra del gráfico por aerolínea | `clickData` | El gráfico de líneas se filtra a esa aerolínea (y vuelve al agregado si se repite el clic) |
| Arrastrar un rectángulo sobre la dispersión | `selectedData` | Aparece el resumen estadístico del subconjunto seleccionado |
| Clic en un punto concreto | `clickData` | Se describe ese único vuelo |
| Pasar el ratón por encima | *hover* | Etiqueta con aerolínea, mes y minutos |

### 13.1 Por qué esto es distinto de los dos tableros anteriores

- **El estímulo no es un control, sino un evento sobre los datos.** `clickData` y `selectedData` son estructuras
  anidadas (`{'points': [{'x': ..., 'y': ..., 'customdata': [...]}]}`), no valores simples. La primera tarea del
  *callback* es **defenderse de la ausencia de selección** (`None`), y la segunda, traducir el evento a una clave
  utilizable (una aerolínea, una lista de índices).
- **Los índices originales hay que transportarlos.** En `Big Data` no se puede enviar el `DataFrame` al navegador;
  se envía `custom_data=['indice']` en la dispersión y el *callback* recupera las filas con `vuelos.loc[...]`.
  Esa es la técnica estándar de *crossfiltering* en Dash.
- **El estado vive en el cliente.** El navegador recuerda qué está seleccionado; el servidor es sin estado. Por
  eso el mismo servidor puede atender a varios usuarios a la vez sin mezclar sus selecciones.


In [59]:
# ===== 13.2 Dashboard 3: datos y figuras de partida =====
# Ranking de aerolineas por retraso medio atribuible (las 12 peores)
promedio_por_aerolinea = (
    vuelos_con_causa
    .groupby('Reporting_Airline')[['CarrierDelay', 'LateAircraftDelay']]
    .mean()
    .assign(total=lambda d: d['CarrierDelay'] + d['LateAircraftDelay'])
    .sort_values('total', ascending=False)
    .head(12)
    .reset_index()
)

figura_barras = px.bar(
    promedio_por_aerolinea, x='Reporting_Airline',
    y=['CarrierDelay', 'LateAircraftDelay'], barmode='group',
    labels={'value': 'Minutos', 'variable': 'Causa'},
    title='Duracion media del retraso por aerolinea (haga clic en una barra)')
figura_barras.for_each_trace(
    lambda traza: traza.update(name=ETIQUETAS_CAUSA.get(traza.name, traza.name)))
figura_barras.update_layout(height=420, legend_title_text='Causa', yaxis_title='Minutos')

# Dispersion de todos los vuelos del rango, con el indice original como dato asociado
dispersion = vuelos.dropna(subset=['DepDelay', 'ArrDelay']).copy()
dispersion['indice'] = dispersion.index
figura_dispersion = px.scatter(
    dispersion, x='DepDelay', y='ArrDelay', color='Reporting_Airline',
    custom_data=['indice'], opacity=0.55,
    labels={'DepDelay': 'Retraso en la salida (min)',
            'ArrDelay': 'Retraso en la llegada (min)'},
    title='Dispersion de retrasos 2010-2020 (arrastre para seleccionar una zona)')
figura_dispersion.update_layout(height=520, legend_title_text='Aerolinea')

print('Aerolineas en el ranking :', len(promedio_por_aerolinea))
print('Vuelos en la dispersion  :', f'{len(dispersion):,}')
print('Ejemplo de customdata    :', figura_dispersion.data[0].customdata[:3])


Aerolineas en el ranking : 12
Vuelos en la dispersion  : 8,740
Ejemplo de customdata    : [[ 1]
 [ 7]
 [13]]


In [60]:
# ===== 13.3 Dashboard 3: layout =====
app3 = dash.Dash('interaccion_cruzada')

app3.layout = html.Div([

    html.H1('Interaccion cruzada: del clic al detalle',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 30}),

    html.P('Haga clic en una barra del primer grafico para filtrar la serie mensual. '
           'En la dispersion, use la herramienta de seleccion por rectangulo de la barra '
           'de herramientas para describir ese subconjunto de vuelos.',
           style={'textAlign': 'center', 'color': '#555555', 'padding': '0 4%'}),

    # --- Segmento 1: barras (fuente del clic) + lineas (respuesta) ---
    html.Div([
        html.Div(dcc.Graph(id='d3-barras', figure=figura_barras,
                           style={'height': '46vh'}), style={'width': '50%'}),
        html.Div(dcc.Graph(id='d3-lineas', style={'height': '46vh'}), style={'width': '50%'}),
    ], style={'display': 'flex', 'padding': '0 1%'}),

    # --- Segmento 2: dispersion (fuente de la seleccion) ---
    html.Div(dcc.Graph(id='d3-dispersion', figure=figura_dispersion,
                       style={'height': '56vh'}), style={'padding': '0 1%'}),

    # --- Segmento 3: resumen del subconjunto seleccionado ---
    html.Div(id='d3-resumen', style={'padding': '0 1% 24px 1%'}),
])

print('Dashboard 3 -> puerto 8052')
print('Componentes con id:', sorted(recorrer(app3.layout)))


Dashboard 3 -> puerto 8052
Componentes con id: ['d3-barras', 'd3-dispersion', 'd3-lineas', 'd3-resumen']


In [61]:
# ===== 13.4 Dashboard 3: callbacks de interaccion =====
@app3.callback(
    Output('d3-lineas', 'figure'),
    Input('d3-barras', 'clickData'),
)
def serie_mensual(clic):
    """Filtra la serie mensual segun la barra en la que se hizo clic.

    OJO: clickData llega como None mientras nadie haya hecho clic, y por eso el
    callback se dispara igualmente al cargar la pagina (llamada inicial).
    """
    if clic:
        aerolinea = clic['points'][0]['x']
        detalle = vuelos_con_causa[vuelos_con_causa['Reporting_Airline'] == aerolinea]
        titulo = f'Evolucion mensual de {aerolinea}'
    else:
        detalle = vuelos_con_causa
        titulo = 'Evolucion mensual (promedio de todas las aerolineas)'

    serie = detalle.groupby('periodo')[VARS_DELAY].mean().reset_index()
    figura = px.line(serie, x='periodo', y=VARS_DELAY, title=titulo)
    figura.for_each_trace(
        lambda traza: traza.update(name=ETIQUETAS_CAUSA.get(traza.name, traza.name)))
    figura.update_layout(height=420, yaxis_title='Minutos', xaxis_title='Mes',
                         legend_title_text='Causa',
                         margin={'l': 50, 'r': 10, 't': 60, 'b': 40})
    return figura


@app3.callback(
    Output('d3-resumen', 'children'),
    Input('d3-dispersion', 'selectedData'),
    Input('d3-dispersion', 'clickData'),
)
def resumen_seleccion(seleccion, clic):
    """Describe el subconjunto de vuelos seleccionado en la dispersion."""
    puntos = (seleccion or {}).get('points') or (clic or {}).get('points')
    if not puntos:
        return html.Div(['Seleccione una zona de la dispersion con la herramienta de rectangulo '
                         'o haga clic en un punto, y apareceran aqui sus estadisticas.'],
                        style={'color': '#777777', 'fontStyle': 'italic'})

    # customdata transporta el indice original de cada vuelo
    indices = [int(p['customdata'][0]) for p in puntos if p.get('customdata')]
    subconjunto = vuelos.loc[indices]
    correlacion = (subconjunto['DepDelay'].corr(subconjunto['ArrDelay'])
                   if len(subconjunto) > 1 else float('nan'))

    return html.Div([
        html.H3(f'Subconjunto seleccionado: {len(subconjunto):,} vuelos'),
        html.Div([
            tarjeta('Retraso medio de salida', f'{subconjunto["DepDelay"].mean():,.1f} min'),
            tarjeta('Retraso medio de llegada', f'{subconjunto["ArrDelay"].mean():,.1f} min'),
            tarjeta('Correlacion salida-llegada', f'{correlacion:.2f}'),
            tarjeta('Aerolineas implicadas', f'{subconjunto["Reporting_Airline"].nunique()}'),
        ], style={'display': 'flex', 'gap': '12px'}),
    ])


print('Callbacks registrados en el dashboard 3:')
for clave in sorted(app3.callback_map):
    print('  -', clave[:70], '...')


Callbacks registrados en el dashboard 3:
  - d3-lineas.figure ...
  - d3-resumen.children ...


In [ ]:
# ===== 13.5 Prueba de los callbacks y puesta en marcha del dashboard 3 =====
figura_sin_clic = serie_mensual(None)
print('Sin clic   -> ', [traza.name for traza in figura_sin_clic.data],
      '| puntos por serie:', len(figura_sin_clic.data[0].x))

figura_con_clic = serie_mensual({'points': [{'x': 'AA'}]})
print('Clic en AA -> ', [traza.name for traza in figura_con_clic.data],
      '| puntos por serie:', len(figura_con_clic.data[0].x))

# Seleccion simulada: los primeros 200 vuelos de la dispersion
indices = dispersion['indice'].head(200).tolist()
seleccion_falsa = {'points': [{'customdata': [int(i)]} for i in indices]}
resumen = resumen_seleccion(seleccion_falsa, None)
print('Resumen de la seleccion ->', resumen.children[0].children)

# Y el caso en que el usuario no ha seleccionado nada
vacio = resumen_seleccion(None, None)
print('Sin seleccion ->', vacio.children[0][:60], '...')

display(lanzar_dashboard(app3, puerto=8052))
display(Markdown('**[Abrir la interaccion cruzada en una pestana](http://127.0.0.1:8052/)**'))


Sin clic   ->  ['Retraso de la aerolinea', 'Clima', 'Sistema aereo nacional', 'Seguridad', 'Aeronave tardia'] | puntos por serie: 123
Clic en AA ->  ['Retraso de la aerolinea', 'Clima', 'Sistema aereo nacional', 'Seguridad', 'Aeronave tardia'] | puntos por serie: 102
Resumen de la seleccion -> Subconjunto seleccionado: 200 vuelos
Sin seleccion -> Seleccione una zona de la dispersion con la herramienta de r ...
El puerto 8052 servia una version anterior del tablero: se reinicia.
Dashboard disponible en http://127.0.0.1:8052/


**[Abrir la interaccion cruzada en una pestana](http://127.0.0.1:8052/)**

127.0.0.1 - - [22/Sep/2026 20:18:08] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:18:08] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_4_1m1790119530.12.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:18:08] "GET /_dash-component-suites/dash/deps/react@18.v4_4_1m1790119530.3.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:18:08] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_4_1m1790119530.3.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:18:08] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_4_1m1790119530.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:18:08] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_4_1m1790119530.8.1.min.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:18:08] "GET /_dash-component-suites/dash/dcc/dash_core_components-shared.v4_4_1m1790119530.js HTTP/1.1" 200 -
127.0.0.1 - - [22/Sep/2026 20:18:08] "GET /_dash-component-suites/dash/dcc/dash_core_components.v4_4_1m1790119530.js HTTP/

---
## 14. Los tres tableros comparados

| | **Tablero 1** (laboratorio) | **Tablero 2** (explorador) | **Tablero 3** (interacción cruzada) |
|---|---|---|---|
| Puerto | 8050 | 8051 | 8052 |
| Controles | 1 × `dcc.Input` | 4 × `Dropdown`, `RangeSlider`, `RadioItems`, `Checklist` + botón | ninguno: el gráfico *es* el control |
| Salidas | 5 × `dcc.Graph` | 4 KPI + `Graph` + `DataTable` + descarga | 3 × `Graph` + 4 KPI |
| Callbacks | 1 (cinco salidas) | 2 | 2 |
| Estímulo | valor de un campo | valor de los controles | `clickData` y `selectedData` |
| Novedad técnica | múltiples `Output` | `State`, `prevent_initial_call`, `dcc.Loading`, `dcc.Download`, columnas dinámicas | eventos sobre los datos y `custom_data` |

### 14.1 Receta para añadir un tablero nuevo

El patrón es siempre el mismo y explica por qué Dash escala bien:

1. **Preparar los datos** una sola vez (`vuelos`, `vuelos_con_causa`) y dejar fuera del *callback* todo lo que no
   dependa de la interacción. Leer el CSV dentro de un *callback* sería el error más caro posible.
2. **Declarar el *layout*** con los componentes y sus `id`; las figuras estáticas pueden asignarse ya en el
   `layout` (como la dispersión del tablero 3) y las dinámicas dejarse vacías.
3. **Escribir un *callback* por bloque lógico**, con todos sus `Output` en una lista, y decidir para cada entrada
   si debe ser `Input` (dispara) o `State` (no dispara).
4. **Probar el *callback* como función pura** (secciones 9.1, 12.5 y 13.5) antes de abrir el navegador.
5. **Servirlo** en un puerto libre con `lanzar_dashboard(app, puerto=...)`.

### 14.2 Advertencia sobre el crecimiento

Estos dos tableros son deliberadamente modestos (8.900 filas en memoria). Con datos grandes el patrón cambia:

- El filtrado debe ocurrir **en la base de datos**, no en pandas, y conviene cachear (`flask_caching`, `Redis`).
- Las devoluciones grandes al navegador se vuelven lentas: se pagina o se agrega en el servidor.
- Los *callbacks* que tardan se envuelven en `dcc.Loading` (aquí ya se usa) o se ejecutan en segundo plano con
  `background_callback`.
- El servidor de desarrollo de Flask no soporta varios usuarios; haría falta un servidor WSGI de producción y,
  si el estado debe compartirse, un almacén externo.

> **Idea de cierre.** La diferencia entre el tablero del laboratorio y el tercero no es de esfuerzo: es de
> **diseño de la interacción**. Los tres usan la misma función auxiliar `compute_info` y el mismo `DataFrame`.


---
## 15. Conclusiones

**Sobre Dash como herramienta de visualización**

1. **El *layout* declarativo escala bien.** Cinco gráficos en tres segmentos se describen con ~20 líneas de
   Python, sin HTML ni JavaScript escrito a mano; reproducir la misma disposición en HTML puro exigiría bastante
   más código y ningún vínculo directo con los datos.
2. **Los *callbacks* separan el cálculo de la presentación.** Todo el análisis vive en `compute_info`, una función
   pura que puede probarse sin servidor; `get_graph` solo la conecta con la interfaz.
3. **Un *callback* de varias salidas evita *callbacks* casi idénticos.** El tablero 1 devuelve cinco figuras de
   una vez; el tablero 2 devuelve indicadores, gráfico y tabla desde los mismos filtros. Filtrar el `DataFrame`
   una sola vez por interacción es lo que mantiene el código legible y rápido.
4. **La validación de entradas es obligatoria.** Sin la guarda de la Tarea 5, borrar el campo del año deja el
   tablero inoperante; sin defensa ante `None` en los controles nuevos, el *callback* del explorador fallaría al
   desmarcar todas las causas.
5. **La interactividad real no está en los controles, sino en el diseño de la consulta.** El tablero 3 no tiene
   ni un solo widget: obtiene más capacidad exploratoria que el tablero 2 porque convierte el clic y la selección
   en filtros.

**Sobre la portabilidad del laboratorio**

6. **El código del laboratorio es portable tal cual.** Se ejecutó sin cambios sustantivos sobre dos entornos
   distintos: el cuaderno con Python 3.13.14 / pandas 2.3.2 / Dash 4.4.1 / Plotly 7.1.0, y la versión en script
   con Python 3.12.10 / pandas 3.0.6 / Dash 4.4.1 / Plotly 7.1.0.
7. **Dos detalles sí requirieron intervención:** la dependencia `httpx==0.20` (exclusiva del Cloud IDE) y la
   construcción de figuras vacías, que `px.line()` sin datos no admite en Plotly 7. Ambos fallos son de
   infraestructura o de versión, no de lógica, y confirman que **registrar las versiones** forma parte del método:
   sin ese registro, un fallo así se atribuye erróneamente al propio código.
8. **Servir Dash desde un cuaderno es viable** con un hilo y un servidor WSGI (`make_server`), y permite tener
   tres tableros simultáneos en tres puertos. Pero el servidor de desarrollo de Flask no es apto para producción:
   para desplegar haría falta un servidor WSGI de producción.

**Sobre el dato y la interpretación**

9. **Un tablero no valida los datos por sí mismo.** La muestra distribuida con el curso tiene 27.000 registros
   frente a los cientos de millones del dataset original, su cobertura mensual es irregular entre 1987 y 2020 y
   el muestreo por aerolínea varía entre 2 y 166 vuelos según el año. El diseño visual es correcto; la inferencia
   estadística es limitada y debe declararse como tal.
10. **El promedio que muestran los tableros es condicional.** El 88,7 % de los registros no tiene causa de
    retraso reportada, de modo que las medias de causas se calculan sobre el 11,3 % de los vuelos. Cada punto es
    la duración media del retraso **cuando esa causa ocurrió**, no un retraso medio por vuelo. Enunciarlo al revés
    es el error de interpretación más fácil de cometer con estos tableros.
11. **La conclusión analítica legítima es descriptiva y comparativa dentro de cada año:** para 2010, el retraso
    por aeronave tardía y el imputable a la aerolínea encabezan los promedios mensuales por aerolínea, y el
    retraso por seguridad es marginal. Cualquier afirmación sobre tendencias entre años, o de tipo causal,
    requeriría la población completa y un diseño muestral conocido.


---
## 16. Limitaciones y extensiones

**Limitaciones de los datos**

- La muestra no es aleatoria por diseño conocido y su cobertura mensual varía por año y aerolínea.
- El 88,7 % de los registros no reporta causas de retraso, por lo que las medias de causas son condicionales y
  descansan sobre 3.057 vuelos.
- Los promedios ignoran la distribución: no hay medidas de dispersión, ni tamaño de grupo, ni intervalos de
  confianza. En 2020 hay celdas mes × aerolínea sostenidas por 2 vuelos.
- Los huecos de las series (promedios `NaN`) se dibujan como cortes, no como ceros.

**Limitaciones técnicas**

- Los tres tableros comparten el `DataFrame` en memoria (~8.900 filas en el rango 2010-2020). Es aceptable para
  el laboratorio y no lo sería para el dataset completo.
- La dispersión dibuja todos los vuelos del rango: con más datos habría que agregar, muestrear o usar WebGL.
- `make_server` / `app.run()` levantan el servidor de desarrollo de Flask, no apto para producción ni para varios
  usuarios simultáneos.
- El diseño no es adaptable (*responsive*): con `display: flex` y anchos fijos, en ventanas estrechas los gráficos
  se comprimen en lugar de apilarse.
- No hay pruebas automatizadas: la verificación se hace invocando los *callbacks* como funciones puras y
  observando el navegador.

**Extensiones propuestas**

| Extensión | Estado |
|---|---|
| Filtrar por aerolínea con `dcc.Dropdown(multi=True)` | ✅ sección 12 |
| Seleccionar el rango de años y usar `State` para no disparar en cada pulsación | ✅ secciones 12 y 13 |
| Comparar aerolíneas con barras agrupadas en vez de 18 líneas | ✅ sección 13 |
| Mediana y conteo de vuelos visibles en el *hover* | pendiente |
| Mapa de calor mes × aerolínea (patrón de estacionalidad de un golpe de vista) | pendiente |
| Comparación A/B de dos años o dos aerolíneas seleccionadas en el propio tablero | pendiente |
| Reunir los tres tableros en uno solo con `dcc.Tabs` y un único `app.py` | pendiente |
| Pruebas automatizadas de la interfaz con `dash.testing` | pendiente |
| Empaquetar con `Procfile` y desplegar en un servidor WSGI de producción | pendiente |

> **Señal de que el diseño es correcto:** las extensiones pendientes se implementan **añadiendo celdas**, no
> reescribiendo las existentes. La separación entre datos, `layout` y *callbacks* es lo que permite que crezca.


---
## 17. Ejercicios (con solución)

### Sobre el tablero del laboratorio

**Ejercicio 1.** Cambie el título del tablero a `Flight Details Statistics Dashboard` con tamaño de fuente 35.

<details>
<summary>Ver solución</summary>

```python
html.H1('Flight Details Statistics Dashboard',
        style={'textAlign': 'center', 'color': '#503D36', 'font-size': 35})
```
</details>

**Ejercicio 2.** El enunciado pide años entre 2010 y 2020. Añada una validación que avise cuando el año esté
fuera de ese rango, sin romper el *callback*.

<details>
<summary>Ver solución</summary>

```python
def get_graph(entered_year):
    try:
        anio = int(entered_year)
    except (TypeError, ValueError):
        anio = None

    if anio is None or not 2010 <= anio <= 2020:
        aviso = Figure()
        aviso.update_layout(title='Indique un anio entre 2010 y 2020',
                            xaxis={'visible': False}, yaxis={'visible': False})
        return [aviso] * 5
    ...
```
</details>

**Ejercicio 3.** Justifique, con una medición sobre los datos, por qué comparar 2010 con 2020 en este tablero es
problemático.

<details>
<summary>Ver solución</summary>

Use la tabla de la sección 11: 2020 solo cubre 3 meses y aporta entre 2 y 39 registros por aerolínea, mientras
que 2010 cubre 12 meses con 9 a 166 registros por aerolínea. El número de puntos por línea es distinto y las
líneas tienen longitudes diferentes: la comparación visual directa no está controlada.
</details>

**Ejercicio 4.** Intercambie el orden de dos `Output` en el decorador y describa qué ocurre en el tablero.

<details>
<summary>Ver solución</summary>

No se produce ningún error: las figuras se asignan por posición, así que los paneles aparecen intercambiados.
Este es el argumento práctico para declarar los `Output` y las devoluciones en el mismo orden.
</details>

### Sobre los tableros interactivos (secciones 12 y 13)

**Ejercicio 5.** Añada al explorador un quinto indicador con la **mediana** del retraso de llegada y explique por
qué no coincide con la media.

<details>
<summary>Ver solución</summary>

En la lista `kpis` de `actualizar_explorador`:

```python
tarjeta('Mediana del retraso de llegada', f"{datos['ArrDelay'].median():,.1f} min"),
```

La mediana es menor que la media porque la distribución de retrasos tiene **cola a la derecha**: la mayoría de
vuelos llega a tiempo o antes, y unos pocos acumulan retrasos enormes (en la sección 5.2 se vio un máximo de 682
minutos). La media es sensible a esos extremos; la mediana no.
</details>

**Ejercicio 6.** Haga que el gráfico de líneas del tablero 3 muestre solo las causas cuyo promedio supere los 10
minutos, sin tocar el gráfico de barras.

<details>
<summary>Ver solución</summary>

La propiedad `figure` de `d3-lineas` la escribe un único *callback*, así que basta filtrar columnas dentro de él:

```python
    serie = detalle.groupby('periodo')[VARS_DELAY].mean().reset_index()
    relevantes = [c for c in VARS_DELAY if serie[c].mean() > 10]
    figura = px.line(serie, x='periodo', y=relevantes, title=titulo)
```
</details>

**Ejercicio 7.** Explique por qué `descargar_csv` recibe los filtros por `State` y no por `Input`, y qué
ocurriría si se cambiaran.

<details>
<summary>Ver solución</summary>

Con `Input`, cada cambio en un control dispararía el *callback* y el navegador descargaría un archivo nuevo cada
vez (al soltar el deslizador, al marcar una casilla...). Con `State`, los valores se leen en el momento en que el
usuario pulsa el botón, que es la única señal que debe provocar una descarga. Es el caso de uso canónico de
`State`: **datos de contexto que no deben disparar nada**.
</details>

**Ejercicio 8.** Añada al resumen de la selección (tablero 3) un indicador con el porcentaje de vuelos
seleccionados cuyo retraso de llegada supera los 15 minutos —el umbral de puntualidad que usa el BTS—.

<details>
<summary>Ver solución</summary>

En la lista de tarjetas de `resumen_seleccion`:

```python
tarjeta('Con mas de 15 min de retraso', f"{(subconjunto['ArrDelay'] > 15).mean():.0%}"),
```

Y para interpretarlo conviene recordar que en 2010 el 88,7 % de los registros de la muestra no tiene causa de
retraso reportada, es decir, son vuelos esencialmente puntuales: el porcentaje obtenido en una selección hecha
sobre la nube inferior de la dispersión será muy pequeño.
</details>


---
## 18. Referencias

1. IBM Developer Skills Network. *DV0101EN — Data Visualization with Python*, laboratorio 4.8
   *Flight Delay Time Statistics Dashboard* (autora: Saishruthi Swaminathan).
2. Plotly. *Dash documentation: Basic callbacks* (incluye la diferencia entre `Input` y `State` y el uso de
   `prevent_initial_call`) <https://dash.plotly.com/basic-callbacks>
3. Plotly. *Dash Core Components — Input, Dropdown, RangeSlider, RadioItems, Checklist, Graph, Loading,
   Download* <https://dash.plotly.com/dash-core-components>
4. Plotly. *Dash DataTable — sorting, filtering and pagination*
   <https://dash.plotly.com/datatable>
5. Plotly. *Dash documentation: Advanced callbacks — `clickData` y `selectedData`*
   <https://dash.plotly.com/advanced-callbacks>
6. Bureau of Transportation Statistics. *Airline Reporting Carrier On-Time Performance* (conjunto de datos
   original del que proviene la muestra).
7. IEEE. *Recommended Practice for Documentation of Computer Programs and Systems* — principio de trazabilidad
   de versiones aplicado en la sección 4.
8. McKinney, W. (2010). *Data Structures for Statistical Computing in Python*. Proceedings of the 9th Python in
   Science Conference (origen de `pandas`).


---
### Ficha de reproducibilidad

| Elemento | Valor |
|---|---|
| Origen del código | Laboratorio 4.8, IBM DV0101EN (módulo 4) |
| Archivo de datos | `airline_data.csv` — 9,8 MB, 27.000 filas × 110 columnas, 33 aerolíneas, 1987-2020 |
| Entorno del cuaderno | Python 3.13.14 · pandas 2.3.2 · Dash 4.4.1 · Plotly 7.1.0 · werkzeug (viene con Dash) |
| Entorno de la versión en script | Python 3.12.10 · pandas 3.0.6 · Dash 4.4.1 · Plotly 7.1.0 |
| Tableros incluidos | 8050 laboratorio · 8051 explorador con filtros · 8052 interacción cruzada |
| Servidores | Hilos demonio creados con `werkzeug.serving.make_server`; se liberan con `detener_dashboard(puerto)` |
| Hallazgo principal sobre los datos | 88,7 % de los registros sin causa de retraso reportada (3.057 de 27.000 con causa) |
| Diferencias respecto del original | datos locales, sin `httpx==0.20`, guarda de entrada, `go.Figure` en lugar de `px.line()` vacío |
| Ampliación respecto del original | secciones 12 a 14: dos tableros nuevos, cinco controles, `State`, `DataTable`, descarga y *crossfiltering* |
| Fecha de verificación | 2026-09-22 |
